In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    precision_recall_curve, 
    average_precision_score
)
from collections import Counter
import matplotlib.pyplot as plt

### Data Loading and Initial Exploration

In [3]:
# Loading the dataset
data = pd.read_csv("creditcard.csv")

# Initial Inspection
print("Shape of dataset:", data.shape, "\n")
print("Column names:", data.columns.tolist())
print("\nDataset Info:")
print(data.describe())

# Class distribution analysis
print("\n🏷️ Class Distribution:")
print(data["Class"].value_counts())
print(f"\nFraud Rate: {(data['Class'].sum() / len(data) * 100):.3f}%")

Shape of dataset: (284807, 31) 

Column names: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Dataset Info:
                Time            V1            V2            V3            V4  \
count  284807.000000  2.848070e+05  2.848070e+05  2.848070e+05  2.848070e+05   
mean    94813.859575  1.168375e-15  3.416908e-16 -1.379537e-15  2.074095e-15   
std     47488.145955  1.958696e+00  1.651309e+00  1.516255e+00  1.415869e+00   
min         0.000000 -5.640751e+01 -7.271573e+01 -4.832559e+01 -5.683171e+00   
25%     54201.500000 -9.203734e-01 -5.985499e-01 -8.903648e-01 -8.486401e-01   
50%     84692.000000  1.810880e-02  6.548556e-02  1.798463e-01 -1.984653e-02   
75%    139320.500000  1.315642e+00  8.037239e-01  1.027196e+00  7.433413e-01   
max    172792.000000  2.454930e+00  2.205773e+01  9.382558e+00  1.687534e+01   

  

### Data Cleaning

In [5]:
# Check for missing values
missing = data.isnull().sum()
if missing.sum() == 0:
    print("No missing values found!")
else:
    print("Missing values per column:")

# Remove duplicates
before_shape = data.shape[0]
data = data.drop_duplicates()
after_shape = data.shape[0]
duplicates_removed = before_shape - after_shape

print(f"\nDuplicates removed: {duplicates_removed:,}")
print(f"Final dataset shape: {data.shape}")

No missing values found!

Duplicates removed: 1,081
Final dataset shape: (283726, 31)


### Feature Selection and Target Preparation

In [7]:
# Prepare features and target
# Remove 'Time' as it's not predictive, keep 'Class' as target
X = data.drop(["Class", "Time"], axis=1, errors="ignore")
y = data["Class"]

print(f"Features shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Check for constant features (quality check)
nunique = X.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
if constant_cols:
    print(f"Dropping constant features: {constant_cols}")
    X = X.drop(columns=constant_cols)
else:
    print("No constant features found")

Features shape: (283726, 29)
Target distribution: {0: 283253, 1: 473}
No constant features found


### Train-Test Split

In [9]:
# Split data (5% for testing to keep training set large)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.05, 
    stratify=y,  # Maintain class distribution
    random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")
print(f"Training fraud rate: {y_train.sum()/len(y_train)*100:.3f}%")

Training set: 269,539 samples
Test set: 14,187 samples
Training fraud rate: 0.167%


### Handling Class Imbalance with SMOTE

In [11]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(f"  Normal: {(y_train == 0).sum():,}")
print(f"  Fraud: {(y_train == 1).sum():,}")

print("\nAfter SMOTE:")
print(f"  Normal: {(y_train_balanced == 0).sum():,}")
print(f"  Fraud: {(y_train_balanced == 1).sum():,}")
print(f"  Total samples: {len(y_train_balanced):,}")

Before SMOTE:
  Normal: 269,090
  Fraud: 449

After SMOTE:
  Normal: 269,090
  Fraud: 269,090
  Total samples: 538,180


### Feature Scaling

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test) 

print("Features scaled successfully!")
print(f"\nTraining features shape: {X_train_scaled.shape}")
print(f"Test features shape: {X_test_scaled.shape}")

Features scaled successfully!

Training features shape: (538180, 29)
Test features shape: (14187, 29)


### SVM Model Training with Hyperparameter Tuning

In [15]:
print("Training SVM model...")

linear_svm = LinearSVC(
    C=1.0, 
    random_state=42, 
    max_iter=2000,
    class_weight='balanced'  # Handle imbalance
)

# Add probability calibration
calibrated_svm = CalibratedClassifierCV(linear_svm, method='sigmoid', cv=3)

# This should take 2-5 minutes instead of 45 minutes
calibrated_svm.fit(X_train_scaled, y_train_balanced)

print("Training completed!")

Training SVM model...
Training completed!


### Making Predictions

In [17]:
print("Making initial predictions...")

y_pred = calibrated_svm.predict(X_test_scaled)
y_pred_proba = calibrated_svm.predict_proba(X_test_scaled)[:, 1]

print("Initial predictions completed")

# Check prediction distribution
print(f"\nPrediction distribution:")
pred_dist = Counter(y_pred)
print(f"Normal: {pred_dist[0]:,} \nFraud: {pred_dist[1]:,}")

Making initial predictions...
Initial predictions completed

Prediction distribution:
Normal: 13,871 
Fraud: 316


### Threshold Optimization

In [19]:
print("Optimizing decision threshold...")

# Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find optimal threshold based on F1 score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\nDefault threshold: 0.5")
print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"\nBest F1 score: {f1_scores[optimal_idx]:.4f}")

# Make predictions with optimal threshold
y_pred_optimized = (y_pred_proba >= optimal_threshold).astype(int)

Optimizing decision threshold...

Default threshold: 0.5
Optimal threshold: 1.0000

Best F1 score: 0.8511


### Model Evaluation

In [21]:
print("\n" + "="*60)
print("                    MODEL EVALUATION")
print("="*60)

# Compare both approaches
approaches = [
    ("Default Threshold (0.5)", y_pred),
    ("Optimized Threshold ({:.3f})".format(optimal_threshold), y_pred_optimized)
]

results_summary = []

for approach_name, predictions in approaches:
    print(f"\n🔍 {approach_name}")
    print("-" * 50)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)
    tn, fp, fn, tp = cm.ravel()
    
    print("Confusion Matrix:")
    print(f"                 Predicted")
    print(f"                Normal  Fraud")
    print(f"Actual Normal   {tn:6}  {fp:5}")
    print(f"       Fraud    {fn:6}  {tp:5}")
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, predictions, target_names=['Normal', 'Fraud'], digits=4))
    
    # Calculate key metrics
    precision_fraud = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_fraud = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_fraud = 2 * (precision_fraud * recall_fraud) / (precision_fraud + recall_fraud) if (precision_fraud + recall_fraud) > 0 else 0
    
    # Store results
    results_summary.append({
        'approach': approach_name,
        'precision': precision_fraud,
        'recall': recall_fraud,
        'f1': f1_fraud,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    })


                    MODEL EVALUATION

🔍 Default Threshold (0.5)
--------------------------------------------------
Confusion Matrix:
                 Predicted
                Normal  Fraud
Actual Normal    13869    294
       Fraud         2     22

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9999    0.9792    0.9894     14163
       Fraud     0.0696    0.9167    0.1294        24

    accuracy                         0.9791     14187
   macro avg     0.5347    0.9480    0.5594     14187
weighted avg     0.9983    0.9791    0.9880     14187


🔍 Optimized Threshold (1.000)
--------------------------------------------------
Confusion Matrix:
                 Predicted
                Normal  Fraud
Actual Normal    14160      3
       Fraud         4     20

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9997    0.9998    0.9998     14163
       Fraud     0.8696    0.8333    0.8511     